In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install split-folders kagglehub ultralytics

import kagglehub
import splitfolders
import os

print("Baixando dataset do Kaggle...")
path = kagglehub.dataset_download("mostafaabla/garbage-classification")
print("Dataset baixado na pasta temporária:", path)

pasta_classes = path
for root, dirs, files in os.walk(path):
    if 'metal' in dirs and 'paper' in dirs:
        pasta_classes = root
        break

print(f"Organizando as imagens a partir de: {pasta_classes}")

splitfolders.ratio(
    pasta_classes,
    output="/content/dataset_yolo",
    seed=42,
    ratio=(0.8, 0.2)
)
print("Dataset organizado com sucesso no formato YOLO!")

In [ ]:
from ultralytics import YOLO

caminho_do_meu_modelo = '/content/drive/MyDrive/EcoSort_Modelos/modelo_classificacao_v2/weights/best.pt'
model = YOLO(caminho_do_meu_modelo)

print("Iniciando o Treinamento Avançado com Foco em Correção de Erros...")

model.train(
    data='/content/dataset_yolo_12_classes',
    epochs=30,
    imgsz=224,
    device=0,
    # Coloca caixas pretas aleatórias em 40% das imagens. Força a IA a olhar o formato inteiro.
    erasing=0.4,
    # HSV_V (Brilho/Value): Altera a iluminação da foto em até 40%.
    hsv_v=0.4,
    # HSV_S (Saturação): Altera a força das cores em até 30%.
    hsv_s=0.3,
    # FLIPLR (Espelhamento Horizontal): Espelha 50% das fotos.
    fliplr=0.5
)

print("\nTreinamento de correção finalizado com sucesso!")
print("Agora seu modelo aprendeu a focar no formato e não apenas no reflexo!")

In [ ]:
from ultralytics import YOLO
from google.colab import files
import IPython.display as display
import json
from datetime import datetime, timezone

caminho_do_modelo = '/content/drive/MyDrive/EcoSort_Modelos/modelo_classificacao_v2/weights/best.pt'
modelo_treinado = YOLO(caminho_do_modelo)

print("Faça o upload de uma imagem:")
uploaded = files.upload()

for nome_arquivo in uploaded.keys():
    caminho_imagem = f"/content/{nome_arquivo}"

    resultados = modelo_treinado.predict(source=caminho_imagem, save=True)

    for resultado in resultados:
        indice_maior_chance = resultado.probs.top1
        nome_da_classe = resultado.names[indice_maior_chance]
        certeza = resultado.probs.top1conf.item()

        payload = {
            "lixeira_id": "smart_bin_01",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "detection": {
                "class_name": nome_da_classe,
                "confidence": round(certeza, 4)
            }
        }

        print(f"\n--- RESULTADO DA CLASSIFICAÇÃO ---")
        print(f"O modelo classificou o resíduo como: {nome_da_classe.upper()}")
        print(f"Certeza de: {certeza * 100:.2f}%\n")

        print("--- JSON GERADO PARA ENVIO À API ---")
        print(json.dumps(payload, indent=2))

        display.display(display.Image(filename=caminho_imagem))